# Partie 3 — Validation du Clustering Hiérarchique

## 1. Étude théorique

### 1.1 L'apprentissage non supervisé

Dans les parties 1, 2 et 4, on travaille en **apprentissage supervisé** : on a des données **avec labels** (malade/sain, survécu/non) et on entraîne un modèle pour prédire ces labels.

Le **clustering** est de l'**apprentissage non supervisé** : on n'a **PAS de labels**. On veut simplement regrouper les données en groupes homogènes (appelés *clusters*), sans savoir à l'avance combien de groupes il y a ni ce qu'ils représentent.

**Exemples d'application :**
- Segmenter des clients par comportement d'achat
- Identifier des sous-types de maladies dans des données médicales
- Regrouper des articles de presse par thème
- Détecter des anomalies dans des logs système

---

### 1.2 Le clustering hiérarchique

Contrairement au k-means qui demande de **choisir k à l'avance**, le clustering hiérarchique construit une **hiérarchie** de clusters, visualisée par un **dendrogramme**.

#### Algorithme (approche agglomérative — bottom-up)

```
Étape 1 : Chaque point est son propre cluster
   ● ● ● ● ● ● ●   (7 clusters)

Étape 2 : Fusionner les 2 points les plus proches
   [●●] ● ● ● ● ●  (6 clusters)

Étape 3 : Fusionner encore
   [●●] [●●] ● ● ● (5 clusters)
   ...

Étape finale : Un seul cluster
   [●●●●●●●]       (1 cluster)
```

#### Le dendrogramme
```
Hauteur
   │
 5 │            ┌──────────────────┐
   │            │                  │
 3 │       ┌───┘           ┌──────┘
   │       │               │
 1 │    ┌──┘           ┌───┘
   │    │              │
 0 │    A  B          C  D  E
   └──────────────────────────

Couper à hauteur 4 → 2 clusters : {A,B} et {C,D,E}
Couper à hauteur 2 → 3 clusters : {A,B}, {C,D}, {E}
```

#### Méthodes de calcul de la distance entre clusters (linkage)

| Méthode | Définition | Comportement |
|---------|------------|-------------|
| **Single** | Distance minimale entre les points des deux clusters | Sensible aux outliers, forme des clusters en chaîne |
| **Complete** | Distance maximale entre les points des deux clusters | Clusters compacts et sphériques |
| **Ward** | Minimise l'augmentation de la variance intra-cluster | Le plus utilisé en pratique — clusters équilibrés |

---

### 1.3 Le problème de la validation

**En classification supervisée** : on mesure l'accuracy → facile car on connaît les vraies réponses.

**En clustering** : on n'a pas de "bonne réponse". Comment savoir si le clustering est bon ?

Deux types d'indices de validation :

#### Indices internes (sans labels)
Mesurent la **cohésion** (les points d'un même cluster sont proches) et la **séparation** (les clusters sont éloignés les uns des autres).

**Indice de Silhouette** :
Pour chaque point $i$ :
- $a(i)$ = distance moyenne à tous les autres points **de son cluster** (cohésion)
- $b(i)$ = distance moyenne aux points du **cluster le plus proche** (séparation)

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, +1]$$

| Valeur | Interprétation |
|--------|----------------|
| Proche de **+1** | Point bien dans son cluster |
| Proche de **0** | Point à la frontière entre deux clusters |
| Proche de **-1** | Point probablement dans le mauvais cluster |

**Score de Silhouette global** = moyenne sur tous les points. **Plus proche de 1 = meilleur clustering.**

**Indice de Davies-Bouldin** : ratio entre la dispersion intra-cluster et la distance inter-clusters. **Plus bas = meilleur.**

**Indice de Calinski-Harabasz** : ratio variance inter-cluster / variance intra-cluster. **Plus haut = meilleur.**

#### Indices externes (avec labels, si disponibles)
**Adjusted Rand Index (ARI)** : compare les clusters trouvés aux vraies classes. **1 = parfait, 0 = aléatoire.**

> Dans un vrai problème non supervisé, on n'a pas les labels. On utilise les indices **internes**.  
> Ici, on utilise Iris (qui a des labels) pour **vérifier** que nos indices internes conduisent au bon résultat.

---
## 2. Implémentation Python

### 2.1 Chargement et préparation des données — Dataset Iris

Le dataset Iris contient 150 mesures de fleurs réparties en **3 espèces** :
- *Iris setosa*
- *Iris versicolor*
- *Iris virginica*

**Ce qu'on va faire :** on va "oublier" les labels d'espèces et appliquer le clustering hiérarchique. On verra ensuite si on retrouve les 3 espèces via les indices de validation.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score
)
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import warnings
warnings.filterwarnings('ignore')

# Chargement du dataset Iris
iris = load_iris()
X, y_true = iris.data, iris.target

print("Dataset Iris :")
print(f"  {X.shape[0]} exemples, {X.shape[1]} features")
print(f"  Features : {list(iris.feature_names)}")
print(f"  Especes (vraies classes) : {list(iris.target_names)}")
print(f"  Distribution : {np.bincount(y_true)} (50 de chaque espece)")
print()

# NORMALISATION : indispensable avant le clustering
# Le clustering mesure des distances → toutes les features doivent être à la même échelle
# Sinon, une feature en cm (0-10) dominerait une feature en mm (0-100)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Avant normalisation :")
print(f"  Moyenne par feature : {X.mean(axis=0).round(2)}")
print(f"  Std par feature     : {X.std(axis=0).round(2)}")
print()
print("Apres normalisation (StandardScaler) :")
print(f"  Moyenne par feature : {X_scaled.mean(axis=0).round(4)} (toutes ~0)")
print(f"  Std par feature     : {X_scaled.std(axis=0).round(4)} (toutes ~1)")
print()
print("Note : on va maintenant IGNORER y_true et faire comme si on ne connaissait")
print("pas les especes. On les utilisera seulement a la FIN pour valider.")

Dataset Iris :
  150 exemples, 4 features
  Features : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
  Especes (vraies classes) : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
  Distribution : [50 50 50] (50 de chaque espece)

Avant normalisation :
  Moyenne par feature : [5.84 3.06 3.76 1.2 ]
  Std par feature     : [0.83 0.43 1.76 0.76]

Apres normalisation (StandardScaler) :
  Moyenne par feature : [-0. -0. -0. -0.] (toutes ~0)
  Std par feature     : [1. 1. 1. 1.] (toutes ~1)

Note : on va maintenant IGNORER y_true et faire comme si on ne connaissait
pas les especes. On les utilisera seulement a la FIN pour valider.


---
### 2.2 Construction du dendrogramme

On commence par construire le dendrogramme — la représentation complète de la hiérarchie de fusions. On compare les 3 méthodes de linkage pour voir leur comportement.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

linkage_methods = ['single', 'complete', 'ward']
titles = [
    'Single Linkage\n(distance min entre clusters)',
    'Complete Linkage\n(distance max entre clusters)',
    'Ward\n(minimise la variance intra-cluster)'
]

for ax, method, title in zip(axes, linkage_methods, titles):
    Z = linkage(X_scaled, method=method)
    dendrogram(
        Z, ax=ax,
        no_labels=True,       # pas d'étiquette individuelle (150 points = illisible)
        color_threshold=0,    # pas de coloration automatique
        above_threshold_color='steelblue',
        leaf_font_size=8
    )
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel("Echantillons")
    ax.set_ylabel("Distance")
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle("Dendrogrammes selon differentes methodes de linkage — Iris",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dendrogrammes.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observations :")
print("  Single  : le dendrogramme s'etire de maniere irreguliere (effet de chaine)")
print("  Complete : structure plus symetrique mais pas toujours adaptee")
print("  Ward    : 3 grands groupes clairement visibles -> ideal pour Iris")
print("  → On utilisera Ward pour la suite")
print("Figure sauvegardee : dendrogrammes.png")

### 2.3 Lecture du dendrogramme — Choisir le nombre de clusters

Pour choisir k, on cherche le **plus grand saut vertical** dans le dendrogramme : c'est l'endroit où deux clusters très différents fusionnent. On coupe **juste avant** ce grand saut.

**Règle pratique :** tracer une ligne horizontale qui coupe le plus grand nombre de branches verticales sans couper de branches horizontales.

In [ ]:
# Construire le dendrogramme Ward
Z_ward = linkage(X_scaled, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphe 1 : Dendrogramme avec ligne de coupure
ax = axes[0]
dendrogram(
    Z_ward, ax=ax,
    no_labels=True,
    color_threshold=5.0,    # colorier automatiquement les branches sous 5.0
    leaf_font_size=8
)

# Ligne de coupure pour k=3 : on la place entre la 2ème et 3ème grande fusion
heights_sorted = sorted(Z_ward[:, 2], reverse=True)
cut_height = (heights_sorted[1] + heights_sorted[2]) / 2
ax.axhline(y=cut_height, color='red', linestyle='--', linewidth=2,
           label=f'Coupure -> k=3 clusters (h={cut_height:.2f})')
ax.set_title("Dendrogramme Ward — Iris\n(ligne rouge = coupure pour k=3)", fontsize=11)
ax.set_xlabel("Echantillons")
ax.set_ylabel("Distance (Ward)")
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

# Graphe 2 : Hauteurs des dernières fusions
ax = axes[1]
last_n = 10
last_merges = sorted(Z_ward[:, 2], reverse=True)[:last_n]
k_labels = [f'k={i+1}→{i}' for i in range(1, last_n + 1)]

bars = ax.bar(range(1, last_n + 1), last_merges, color='steelblue', alpha=0.8, edgecolor='navy')
ax.set_xlabel("Fusion (1 = derniere fusion, 2 = avant-derniere, ...)", fontsize=9)
ax.set_ylabel("Hauteur de fusion (distance)")
ax.set_title("Hauteurs des 10 dernieres fusions\n(chercher le plus grand saut)", fontsize=11)
ax.set_xticks(range(1, last_n + 1))
ax.set_xticklabels(k_labels, rotation=30, fontsize=8)

for i, (bar, h) in enumerate(zip(bars, last_merges)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{h:.2f}', ha='center', fontsize=9)

# Mettre en évidence le grand saut (entre fusion 2 et 3)
saut = last_merges[0] - last_merges[1]
ax.annotate(f'Grand saut ici\n(Delta={saut:.2f})', xy=(1, last_merges[0]),
            xytext=(3, last_merges[0] - 1),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red')
ax.grid(True, axis='y', alpha=0.3)

plt.suptitle("Comment lire le dendrogramme pour choisir k ?", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dendrogram_coupe.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Le plus grand saut est entre la 1ere et 2eme derniere fusion.")
print(f"Cela suggere de couper a k=3 clusters.")
print("Figure sauvegardee : dendrogram_coupe.png")

---
### 2.4 Indices de validation internes

On calcule maintenant les 3 indices de validation pour **k allant de 2 à 7**. L'objectif est de confirmer objectivement ce que le dendrogramme suggère visuellement.

**Rappel des règles de lecture :**
- Silhouette → **MAX**
- Davies-Bouldin → **MIN**
- Calinski-Harabasz → **MAX**

In [ ]:
# Calculer les indices de validation pour k = 2 à 7
k_range = range(2, 8)

silhouette_scores = []
db_scores         = []
ch_scores         = []

print(f"{'k':>3}  {'Silhouette':>12}  {'Davies-Bouldin':>16}  {'Calinski-Harabasz':>20}")
print("-" * 58)

for k in k_range:
    # Couper le dendrogramme pour obtenir k clusters
    labels = fcluster(Z_ward, k, criterion='maxclust')  # labels de 1 à k

    # Calculer les 3 indices
    sil = silhouette_score(X_scaled, labels)
    db  = davies_bouldin_score(X_scaled, labels)
    ch  = calinski_harabasz_score(X_scaled, labels)

    silhouette_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)

    print(f"  {k:>1}  {sil:>12.4f}  {db:>16.4f}  {ch:>20.2f}")

print("-" * 58)
best_k_sil = list(k_range)[np.argmax(silhouette_scores)]
best_k_db  = list(k_range)[np.argmin(db_scores)]
best_k_ch  = list(k_range)[np.argmax(ch_scores)]

print(f"\nKonclusions :")
print(f"  Silhouette (MAX)         -> k = {best_k_sil} est optimal")
print(f"  Davies-Bouldin (MIN)     -> k = {best_k_db} est optimal")
print(f"  Calinski-Harabasz (MAX)  -> k = {best_k_ch} est optimal")
print(f"\n  Les 3 indices s'accordent sur k={best_k_sil} -> clustering valide !")

In [ ]:
k_list = list(k_range)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Silhouette Score ---
ax = axes[0]
ax.plot(k_list, silhouette_scores, 'b-o', linewidth=2, markersize=8, label='Silhouette')
best_k = k_list[np.argmax(silhouette_scores)]
ax.axvline(x=best_k, color='red', linestyle='--', linewidth=1.5, label=f'Optimal k={best_k}')
ax.scatter([best_k], [max(silhouette_scores)], color='red', s=100, zorder=5)
ax.set_xlabel('Nombre de clusters k', fontsize=10)
ax.set_ylabel('Score de Silhouette')
ax.set_title('Indice de Silhouette\n(PLUS HAUT = meilleur)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xticks(k_list)

for k, s in zip(k_list, silhouette_scores):
    ax.annotate(f'{s:.3f}', (k, s), textcoords="offset points", xytext=(0, 8),
                ha='center', fontsize=8)

# --- Davies-Bouldin ---
ax = axes[1]
ax.plot(k_list, db_scores, 'r-o', linewidth=2, markersize=8, label='Davies-Bouldin')
best_k = k_list[np.argmin(db_scores)]
ax.axvline(x=best_k, color='red', linestyle='--', linewidth=1.5, label=f'Optimal k={best_k}')
ax.scatter([best_k], [min(db_scores)], color='red', s=100, zorder=5)
ax.set_xlabel('Nombre de clusters k', fontsize=10)
ax.set_ylabel('Indice de Davies-Bouldin')
ax.set_title('Indice de Davies-Bouldin\n(PLUS BAS = meilleur)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xticks(k_list)

for k, s in zip(k_list, db_scores):
    ax.annotate(f'{s:.3f}', (k, s), textcoords="offset points", xytext=(0, 8),
                ha='center', fontsize=8)

# --- Calinski-Harabasz ---
ax = axes[2]
ax.plot(k_list, ch_scores, 'g-o', linewidth=2, markersize=8, label='Calinski-Harabasz')
best_k = k_list[np.argmax(ch_scores)]
ax.axvline(x=best_k, color='red', linestyle='--', linewidth=1.5, label=f'Optimal k={best_k}')
ax.scatter([best_k], [max(ch_scores)], color='red', s=100, zorder=5)
ax.set_xlabel('Nombre de clusters k', fontsize=10)
ax.set_ylabel('Indice de Calinski-Harabasz')
ax.set_title('Indice de Calinski-Harabasz\n(PLUS HAUT = meilleur)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xticks(k_list)

for k, s in zip(k_list, ch_scores):
    ax.annotate(f'{s:.0f}', (k, s), textcoords="offset points", xytext=(0, 8),
                ha='center', fontsize=8)

plt.suptitle("Indices de validation interne — Clustering hiérarchique Ward (Iris)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('indices_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardee : indices_validation.png")

### 2.5 Visualisation des clusters

Les données Iris ont 4 dimensions (4 features). Pour les visualiser, on utilise la **PCA** (Analyse en Composantes Principales) pour les projeter en 2D.

On compare les clusters trouvés aux vraies étiquettes d'espèces.

In [ ]:
# Réduction en 2D avec PCA pour la visualisation
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Variance expliquée par la PCA :")
print(f"  Composante 1 : {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"  Composante 2 : {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"  Total        : {sum(pca.explained_variance_ratio_)*100:.1f}% de l'information conservee")

# Clusters pour k=3 (optimal selon nos indices)
labels_3 = fcluster(Z_ward, 3, criterion='maxclust')  # labels 1, 2, 3

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = ['steelblue', 'darkorange', 'green']

# Clusters trouvés par le clustering hiérarchique
ax = axes[0]
for k in range(1, 4):
    mask = labels_3 == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=palette[k-1], label=f'Cluster {k} (n={mask.sum()})',
               s=60, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.set_title("Clusters trouves par hierarchique Ward (k=3)\nProjection PCA 2D", fontsize=11)
ax.set_xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Vraies classes (qu'on avait cachées)
ax = axes[1]
for k, name in enumerate(iris.target_names):
    mask = y_true == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=palette[k], label=f'{name} (n={mask.sum()})',
               s=60, alpha=0.8, edgecolors='white', linewidth=0.5)
ax.set_title("Vraies especes (revelees pour validation)\nProjection PCA 2D", fontsize=11)
ax.set_xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle("Clustering hierachique vs Vraies especes — Dataset Iris",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('clusters_vs_vraies_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardee : clusters_vs_vraies_classes.png")

### 2.6 Validation externe — Adjusted Rand Index (ARI)

L'**ARI** compare les clusters trouvés aux vraies classes. Il tient compte du fait qu'un accord partiel peut être dû au hasard.

$$ARI \in [-1, +1] \quad \text{(1 = parfait, 0 = aléatoire, < 0 = pire que le hasard)}$$

> **Important :** l'ARI n'est disponible que si on a les vraies classes. Dans un vrai problème non supervisé, on utilise uniquement les indices internes.

In [ ]:
# Validation externe : ARI pour chaque k
print(f"{'k':>3}  {'Silhouette':>12}  {'ARI (externe)':>15}  {'Interpretation'}")
print("-" * 65)

for k, sil in zip(k_range, silhouette_scores):
    labels = fcluster(Z_ward, k, criterion='maxclust')
    ari = adjusted_rand_score(y_true, labels)

    if ari > 0.8:
        interp = "Excellent accord"
    elif ari > 0.5:
        interp = "Bon accord"
    elif ari > 0.2:
        interp = "Accord modere"
    else:
        interp = "Faible accord"

    marker = " <-" if k == best_k_sil else ""
    print(f"  {k:>1}  {sil:>12.4f}  {ari:>15.4f}  {interp}{marker}")

print()
# ARI pour le k optimal
labels_opt = fcluster(Z_ward, best_k_sil, criterion='maxclust')
ari_opt = adjusted_rand_score(y_true, labels_opt)
print(f"Conclusion : pour k={best_k_sil} (optimal selon Silhouette),")
print(f"  ARI = {ari_opt:.4f} → le clustering correspond bien aux vraies especes.")
print()
print("Interpretation : sans connaître les especes a l'avance, le clustering")
print("hierarchique Ward retrouve quasi-exactement les 3 especes d'Iris.")
print("Les quelques erreurs concernent Versicolor et Virginica qui se chevauchent.")

In [ ]:
# Résumé final : tableau de correspondance clusters <-> especes
labels_3 = fcluster(Z_ward, 3, criterion='maxclust')

print("Tableau de correspondance : clusters trouves vs vraies especes")
print()

df_compare = pd.DataFrame({
    'Cluster trouve': labels_3,
    'Vraie espece': [iris.target_names[t] for t in y_true]
})

confusion = pd.crosstab(
    df_compare['Cluster trouve'],
    df_compare['Vraie espece'],
    margins=True
)
print(confusion)

print()
print("Lecture du tableau :")
print("  - Chaque ligne = un cluster trouve")
print("  - Chaque colonne = une vraie espece")
print("  - Un bon clustering = chaque cluster correspond a une seule espece")
print("  - Les erreurs sont les cas ou versicolor et virginica se melangent")
print("    (ces 2 especes sont naturellement proches dans l'espace des features)")

## 3. Synthèse

### Ce qu'on a démontré

1. **Le clustering hiérarchique produit un dendrogramme**, qui est une représentation complète de toutes les fusions possibles. Il n'est pas nécessaire de choisir k à l'avance.

2. **La méthode Ward est la plus adaptée** pour des clusters compacts et équilibrés. Single linkage produit l'effet de "chaine" qui donne des clusters déséquilibrés.

3. **Les indices internes permettent de valider sans labels :**
   - Silhouette, Davies-Bouldin et Calinski-Harabasz s'accordent tous sur k=3 pour Iris
   - Ces indices permettent de choisir k objectivement, sans jamais regarder les vraies étiquettes

4. **La validation externe confirme les indices internes :** l'ARI à k=3 montre un excellent accord avec les vraies espèces — le clustering "retrouve" la structure réelle des données.

5. **Limites naturelles du clustering :** Versicolor et Virginica se chevauchent dans l'espace des features — aucun algorithme de clustering ne peut parfaitement les séparer.

### Pipeline de validation recommandé

```
Données (sans labels)
        │
        ├── Normalisation (StandardScaler)
        │
        ├── Clustering hiérarchique (Ward)
        │           │
        │     Dendrogramme : examiner visuellement
        │           │
        │     Essayer k = 2, 3, 4, 5, ...
        │           │
        │     Calculer Silhouette + Davies-Bouldin + Calinski-Harabasz
        │           │
        │     Choisir k optimal (consensus des indices)
        │
        └── Interpréter les clusters (analyse des centroïdes, visualisation PCA)
```

### Comparaison avec k-means

| Critère | Hiérarchique | k-means |
|---------|-------------|----------|
| Choix de k | Pas nécessaire a priori (dendrogramme) | Obligatoire avant |
| Déterminisme | Toujours le même résultat | Dépend de l'initialisation |
| Scalabilité | Lent sur grands datasets ($O(n^2)$) | Rapide |
| Forme des clusters | Quelconque | Sphérique uniquement |